# Calculate the amplitudes

This notebook is used to calculate the amplitudes of the seismograms.

Agentic AI was used in this notebook.

By Hiroto Bito

In [22]:
# Import necessary libraries
import os
import sys
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

from obspy import read, UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.stream import Stream

parent_dir = '/home/hbito/cascadia_obs_ensemble/utils'
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from data_client import get_waveforms

In [23]:
# Read the data frame
datasets_dir =  '/wd1/hbito_data/data/datasets_all_regions'
path_assigned_picks_df = f'{datasets_dir}/Cascadia_updated_catalog_picks_assignment_ver_3.csv'

# Prepare output CSV path 
output_csv_path = f'{datasets_dir}/Cascadia_updated_catalog_picks_assignment_ver_3_w_amp_test.csv'

# File to save skipped picks
skipped_csv_path = f'{datasets_dir}/calculate_amplitudes_skipped_picks_test.csv'

assigned_picks_original_df = pd.read_csv(path_assigned_picks_df, index_col=False).copy()

test

In [15]:
assigned_picks_original_df

,Unnamed: 0,time_pick,station,phase,timeres,idx,arid,latitude,longitude,depth,Detection Value,time,RMS Residual (s),Num. P,Num. S,picks,slatitude,slongitude,selevation
0,0,2010-01-01T00:15:27.180000Z,UW.PCMD,P,0.049,0,0,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0
1,1,2010-01-01T00:15:37.840400Z,UW.RVW,P,1.264,0,1,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.149750,-122.742996,504.0
2,2,2010-01-01T00:15:33.280000Z,UW.PCMD,S,-0.243,0,2,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0
3,3,2010-01-01T00:15:42.002000Z,UW.GNW,S,2.402,0,3,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.564130,-122.824980,220.0
4,4,2010-01-01T00:15:43.618400Z,PB.B013,S,-0.651,0,4,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.813000,-122.910797,75.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1004330,1004330,2015-06-23T23:18:40.885798Z,7D.J11D,S,0.044,63886,1004330,43.33308,-127.31240,6.0615,0.694,2015-06-23 23:18:19.457000+00:00,0.447,4,5,9,43.541599,-126.368599,-3000.8
1004331,1004331,2015-06-23T23:18:48.573898Z,7D.G35D,S,0.358,63886,1004331,43.33308,-127.31240,6.0615,0.694,2015-06-23 23:18:19.457000+00:00,0.447,4,5,9,42.555698,-126.399002,-2822.6
1004332,1004332,2015-06-23T23:18:50.458298Z,7D.J19D,S,0.300,63886,1004332,43.33308,-127.31240,6.0615,0.694,2015-06-23 23:18:19.457000+00:00,0.447,4,5,9,44.179001,-126.271202,-2955.4
1004333,1004333,2015-06-23T23:18:56.689277Z,7D.J10D,S,0.432,63886,1004333,43.33308,-127.31240,6.0615,0.694,2015-06-23 23:18:19.457000+00:00,0.447,4,5,9,43.348499,-125.545097,-3085.0


In [16]:
type(assigned_picks_original_df.iloc[0]['time_pick'])

str

In [17]:
pick_time_str = assigned_picks_original_df.iloc[0]['time_pick']

In [18]:
pick_time_utcdatetime = UTCDateTime(pick_time_str)
pick_time_utcdatetime

2010-01-01T00:15:27.180000Z

In [20]:
starttime = pick_time_utcdatetime - 0.5
starttime

2010-01-01T00:15:26.680000Z

In [21]:
type(starttime)

obspy.core.utcdatetime.UTCDateTime

end test

In [24]:
# Randomly pick a subset of the table
n_picks = 500
assigned_picks_df = assigned_picks_original_df.iloc[:n_picks].copy()
assigned_picks_df

,Unnamed: 0,time_pick,station,phase,timeres,idx,arid,latitude,longitude,depth,Detection Value,time,RMS Residual (s),Num. P,Num. S,picks,slatitude,slongitude,selevation
0,0,2010-01-01T00:15:27.180000Z,UW.PCMD,P,0.049,0,0,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0
1,1,2010-01-01T00:15:37.840400Z,UW.RVW,P,1.264,0,1,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.149750,-122.742996,504.0
2,2,2010-01-01T00:15:33.280000Z,UW.PCMD,S,-0.243,0,2,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0
3,3,2010-01-01T00:15:42.002000Z,UW.GNW,S,2.402,0,3,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.564130,-122.824980,220.0
4,4,2010-01-01T00:15:43.618400Z,PB.B013,S,-0.651,0,4,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.813000,-122.910797,75.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,495,2010-01-02T18:19:35.130100Z,UW.LCW,P,-0.491,21,495,46.20482,-122.81132,41.3420,0.604,2010-01-02 18:19:24.788000+00:00,0.473,8,1,9,46.670490,-122.702011,396.0
496,496,2010-01-02T18:19:35.140100Z,UW.MTM,P,-0.399,21,496,46.20482,-122.81132,41.3420,0.604,2010-01-02 18:19:24.788000+00:00,0.473,8,1,9,46.025330,-122.212870,1121.0
497,497,2010-01-02T18:19:36.490100Z,UW.RVW,S,-0.166,21,497,46.20482,-122.81132,41.3420,0.604,2010-01-02 18:19:24.788000+00:00,0.473,8,1,9,46.149750,-122.742996,504.0
498,498,2010-01-02T19:18:22.568400Z,PB.B943,S,-0.775,22,498,47.74010,-123.12926,9.3365,0.680,2010-01-02 19:18:16.657000+00:00,0.514,0,6,6,47.813202,-122.911301,84.2


In [25]:
# Define the arguments
window_before = 0.5 # in sec
window_after = 2 # in sec
source = 'pnwstore'

freq_highpass = 2 # in Hz
new_sampling_rate = 100 # in Hz

In [ ]:
# Run the loop
amplitudes = []

for idx, row in assigned_picks_df.iterrows():

    # Define the arguments 
    date, _time = row['time'].split(' ')
    datetime_str = date+'T'+_time
    origin_time = UTCDateTime(datetime_str)  # Accept ISO string directly

    time_pick_str = row['time_pick'] 
    time_pick = UTCDateTime(time_pick_str)  # Accept ISO string directly

    network = row['station'].split('.')[0].strip()
    station = row['station'].split('.')[1].strip()
    channel = '*H*'
    starttime = time_pick - window_before 
    endtime = time_pick + window_after
   

    # # Print the number of items in amplitudes
    # print('len(amplitudes)',len(amplitudes))    

    # Request a waveform
    time.sleep(0.1)

    try:
        st = get_waveforms(network=network, station=station, channel=channel,
                            starttime=starttime, endtime=endtime,
                            source=source)
    except Exception as e:
        print(f"Request failed: {e}")

        # Save amplitude to the output DataFrame and CSV on the fly
        amp = np.nan
        amplitudes.append(amp)

        # Save skipped info to CSV
        skipped_info = {
            'network': network,
            'station': station,
            'channel': channel,
            'origin_time': origin_time,
            'time_pick': time_pick,
            'starttime': starttime,
            'endtime': endtime,
            'reason': f'Request failed: {e}'
        }
        df_skipped = pd.DataFrame([skipped_info])
        if not os.path.isfile(skipped_csv_path):
            df_skipped.to_csv(skipped_csv_path, mode='w', header=True, index=False)
        else:
            df_skipped.to_csv(skipped_csv_path, mode='a', header=False, index=False)

        continue
        

    # time.sleep(0.1)


    # Create a new stream
    sdata = Stream()
    
    # Check if loaded data have a vertical component (minimum requirement)
    has_Z = bool(st.select(id=f'{network}.{station}..??Z'))
    # Check for the presence of HH, BH, and EH channels
    has_HH = bool(st.select(id=f'{network}.{station}..HH?'))
    has_BH = bool(st.select(id=f'{network}.{station}..BH?'))
    has_EH = bool(st.select(id=f'{network}.{station}..EH?'))

    if not has_Z:
        e = f'No Vertical Component Data Present at {network}.{station} with HHZ, BHZ or EHZ channels at {time_pick_str}. Skipping'
        print(e)

        # Save amplitude in the list
        amp = np.nan
        amplitudes.append(amp)

        # Save skipped info to CSV
        skipped_info = {
            'network': network,
            'station': station,
            'channel': channel,
            'origin_time': origin_time,
            'time_pick': time_pick,
            'starttime': starttime,
            'endtime': endtime,
            'reason': f'Request failed: {e}'
        }
        df_skipped = pd.DataFrame([skipped_info])
        if not os.path.isfile(skipped_csv_path):
            df_skipped.to_csv(skipped_csv_path, mode='w', header=True, index=False)
        else:
            df_skipped.to_csv(skipped_csv_path, mode='a', header=False, index=False)

        continue

    # Apply selection logic based on channel presence
    if has_HH:
        # If all HH, BH, and EH, channels are present, select only HH
        sdata += st.select(id=f'{network}.{station}..HH*')
    elif has_BH:
        # If BH and EH channels are present, select only BH
        sdata += st.select(id=f'{network}.{station}..BH*')
    elif has_EH:
        # If only EH channels are present, select only EH
        # NTS: This may result in getting only vertical component data - EH? is used for PNSN analog stations
        # NTS: This may also be tricky for pulling full day-volumes because the sampling rate shifts for
        #      analog stations due to the remote digitization scheme used with analog stations
        sdata += st.select(id=f'{network}.{station}..EH*')
    else:
        e = f'No data available at {network}.{station} with HHZ, BHZ or EHZ channels at {time_pick_str}. Skipping.'
        print(e)

        # Save amplitude to the output DataFrame and CSV on the fly
        amp = np.nan
        amplitudes.append(amp)

        # Save skipped info to CSV
        skipped_info = {
            'network': network,
            'station': station,
            'channel': channel,
            'origin_time': origin_time,
            'time_pick': time_pick,
            'starttime': starttime,
            'endtime': endtime,
            'reason': f'Request failed: {e}'
        }
        df_skipped = pd.DataFrame([skipped_info])
        if not os.path.isfile(skipped_csv_path):
            df_skipped.to_csv(skipped_csv_path, mode='w', header=True, index=False)
        else:
            df_skipped.to_csv(skipped_csv_path, mode='a', header=False, index=False)
        continue

    # Resample
    sdata.resample(new_sampling_rate)
        
    # Apply highpass filter
    sdata.detrend(type='demean')
    sdata.taper(max_percentage=0.05)
    sdata.filter(type='highpass', freq=freq_highpass)

    max_amp = 0
    for tr in sdata:
        max_amp = max(max_amp, abs(tr.data).max())

    amplitudes.append(max_amp)

# Append to DataFrame
assigned_picks_df.loc[:,"Amplitude"] = amplitudes

assigned_picks_df.to_csv(output_csv_path, index=False)

  0%|          | 0/500 [00:00<?, ?it/s]

len(amplitudes) 0


PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
  0%|          | 1/500 [00:00<07:10,  1.16it/s]PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09


len(amplitudes) 1


  1%|          | 3/500 [00:01<02:48,  2.96it/s]

...

PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
100%|█████████▉| 498/500 [01:47<00:00,  7.05it/s]

len(amplitudes) 497
len(amplitudes) 498


PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09
100%|█████████▉| 499/500 [01:47<00:00,  6.18it/s]PNWstore | WARNING | Missing mount for year 2023 at /auto/pnwstore1-wd09


len(amplitudes) 499


100%|██████████| 500/500 [01:47<00:00,  4.65it/s]


In [31]:
test = pd.read_csv(output_csv_path)
test

,Unnamed: 0,time_pick,station,phase,timeres,idx,arid,latitude,longitude,depth,Detection Value,time,RMS Residual (s),Num. P,Num. S,picks,slatitude,slongitude,selevation,Amplitude
0,0,2010-01-01T00:15:27.180000Z,UW.PCMD,P,0.049,0,0,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0,534.658499
1,1,2010-01-01T00:15:37.840400Z,UW.RVW,P,1.264,0,1,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.149750,-122.742996,504.0,73.403575
2,2,2010-01-01T00:15:33.280000Z,UW.PCMD,S,-0.243,0,2,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,46.888962,-122.301483,239.0,897.182138
3,3,2010-01-01T00:15:42.002000Z,UW.GNW,S,2.402,0,3,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.564130,-122.824980,220.0,2239.571065
4,4,2010-01-01T00:15:43.618400Z,PB.B013,S,-0.651,0,4,47.13396,-122.09098,60.1470,0.680,2010-01-01 00:15:16.204000+00:00,1.081,2,5,7,47.813000,-122.910797,75.3,60.932284
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,495,2010-01-02T18:19:35.130100Z,UW.LCW,P,-0.491,21,495,46.20482,-122.81132,41.3420,0.604,2010-01-02 18:19:24.788000+00:00,0.473,8,1,9,46.670490,-122.702011,396.0,583.094788
496,496,2010-01-02T18:19:35.140100Z,UW.MTM,P,-0.399,21,496,46.20482,-122.81132,41.3420,0.604,2010-01-02 18:19:24.788000+00:00,0.473,8,1,9,46.025330,-122.212870,1121.0,973.045194
497,497,2010-01-02T18:19:36.490100Z,UW.RVW,S,-0.166,21,497,46.20482,-122.81132,41.3420,0.604,2010-01-02 18:19:24.788000+00:00,0.473,8,1,9,46.149750,-122.742996,504.0,842.538768
498,498,2010-01-02T19:18:22.568400Z,PB.B943,S,-0.775,22,498,47.74010,-123.12926,9.3365,0.680,2010-01-02 19:18:16.657000+00:00,0.514,0,6,6,47.813202,-122.911301,84.2,1281.701150
